# **Performance python**

## Loops

A function calls a certain large number of random numbers and returns their average

### Python

In [93]:
import random

In [94]:
def average_py(n):
    s = 0
    for i in range(n):
        s += random.random()
    return s / n

In [95]:
n = 10000000
%time average_py(n)

CPU times: user 2.05 s, sys: 3.85 ms, total: 2.06 s
Wall time: 2.06 s


0.49988943812644204

In [96]:
%timeit average_py(n)

1.14 s ± 88.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [97]:
%time sum([random.random() for _ in range(n)]) / n

CPU times: user 1.72 s, sys: 315 ms, total: 2.04 s
Wall time: 2.04 s


0.4999619837449639

### Numpy

In [98]:
import numpy as np

In [99]:
def average_np(n):
    s = np.random.random(n)
    return s.mean()

In [100]:
%time average_np(n)

CPU times: user 167 ms, sys: 9.02 ms, total: 176 ms
Wall time: 175 ms


np.float64(0.5000040939583701)

In [101]:
%timeit average_np(n)

152 ms ± 1.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [102]:
s = np.random.random(n)
s.nbytes

80000000

### Numba

In [103]:
import numba

In [104]:
average_nb = numba.jit(average_py)

In [105]:
%time average_nb(n)

CPU times: user 369 ms, sys: 0 ns, total: 369 ms
Wall time: 368 ms


0.5000894867391652

In [106]:
%timeit average_nb(n)

245 ms ± 9.04 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Cython

In [107]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [108]:
%%cython -a
import random
def average_cy1(int n):
    cdef int i
    cdef float s = 0
    for i in range(n):
        s += random.random()
    return s / n

In [109]:
%time average_cy1(n)

CPU times: user 1.64 s, sys: 1.24 ms, total: 1.64 s
Wall time: 1.62 s


0.4999716579914093

In [110]:
%timeit average_cy1(n)

1.57 s ± 33.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [111]:
%%cython
from libc.stdlib cimport rand
cdef extern from "limits.h":
    int INT_MAX
cdef int i
cdef float rn
for i in range(5):
    rn = rand() / INT_MAX
    print(rn)

In [112]:
%%cython -a
from libc.stdlib cimport rand
cdef extern from "limits.h":
    int INT_MAX
def average_cy2(int n):
    cdef int i
    cdef float s = 0
    for i in range(n):
        s += rand() / INT_MAX
    return s / n

In [113]:
%time average_cy2(n)

CPU times: user 373 ms, sys: 1.03 ms, total: 374 ms
Wall time: 370 ms


0.4999966621398926

In [114]:
%timeit average_cy2(n)

419 ms ± 100 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Algorithms 

Applying performance enabling techniques to well known problems and algorithms 

### Primce numbers

In [115]:
def is_prime(I):
    if I < 2: return False
    for i in range(2, int(I ** 0.5) + 1):
        if I % i == 0: return False
    return True

In [116]:
n = int(1e8+3)

In [117]:
%time is_prime(n)

CPU times: user 223 μs, sys: 0 ns, total: 223 μs
Wall time: 236 μs


False

In [118]:
p1 = int(1e8 + 7)
p1

100000007

In [119]:
%time is_prime(p1)

CPU times: user 1.91 ms, sys: 0 ns, total: 1.91 ms
Wall time: 1.92 ms


True

In [120]:
p2 = 100109100129162907
p2.bit_length()

57

In [121]:
%time is_prime(p2)

CPU times: user 59.3 s, sys: 2.35 ms, total: 59.3 s
Wall time: 59.5 s


True

### Numba

In [122]:
is_prime_nb = numba.jit(is_prime)

In [123]:
%time is_prime_nb(n)

CPU times: user 131 ms, sys: 2.99 ms, total: 134 ms
Wall time: 133 ms


False

In [124]:
%time is_prime_nb(n)

CPU times: user 20 μs, sys: 0 ns, total: 20 μs
Wall time: 26.9 μs


False

In [125]:
%time is_prime_nb(p1)

CPU times: user 134 μs, sys: 2 μs, total: 136 μs
Wall time: 147 μs


True

In [126]:
%time is_prime_nb(p2)

CPU times: user 6.7 s, sys: 41.3 ms, total: 6.74 s
Wall time: 6.65 s


True

### Cython

In [127]:
%%cython
def is_prime_cy1(I):
    if I < 2: return False
    for i in range(2, int(I ** 0.5) + 1):
        if I % i == 0: return False
    return True

In [128]:
%timeit is_prime(p1)

1.25 ms ± 92.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [129]:
%timeit is_prime_cy1(p1)

960 μs ± 187 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [130]:
%%cython
def is_prime_cy2(long I):
    cdef long i
    if I < 2: return False
    for i in range(2, int(I ** 0.5) + 1):
        if I % i == 0: return False
    return True

In [131]:
%time is_prime_cy2(p1)

CPU times: user 201 μs, sys: 2 μs, total: 203 μs
Wall time: 210 μs


True

In [132]:
%time is_prime_nb(p2)

CPU times: user 6.42 s, sys: 29.3 ms, total: 6.44 s
Wall time: 6.37 s


True

In [133]:
%time is_prime_cy2(p2)

CPU times: user 6.13 s, sys: 26.5 ms, total: 6.15 s
Wall time: 6.08 s


True

### Multiprocessing

In [134]:
import multiprocessing as mp

In [139]:
pool = mp.Pool(processes=4)

In [136]:
%time pool.map(is_prime, 10*[p1])

CPU times: user 6.42 ms, sys: 7 ms, total: 13.4 ms
Wall time: 19.4 ms


[True, True, True, True, True, True, True, True, True, True]

In [140]:
%time pool.map(is_prime_nb, 10*[p2])

CPU times: user 15.1 ms, sys: 5.96 ms, total: 21 ms
Wall time: 38.3 s


[True, True, True, True, True, True, True, True, True, True]

In [141]:
%time pool.map(is_prime_cy2, 10 * [p2])

CPU times: user 7.34 ms, sys: 4.98 ms, total: 12.3 ms
Wall time: 23.9 s


[True, True, True, True, True, True, True, True, True, True]

## Fibonacci Numbers

### Recursive Algorithm

In [189]:
def fib_rec_py1(n):
    if n < 2:
        return n
    else:
        return fib_rec_py1(n-1) + fib_rec_py1(n-2)

In [190]:
%time fib_rec_py1(35)

CPU times: user 4.42 s, sys: 5.88 ms, total: 4.43 s
Wall time: 4.43 s


9227465

In [187]:
%%cython
def fib_rec_cy(int n):
    if n < 2:
        return n
    else:
        return fib_rec_cy(n-1) + fib_rec_cy(n-2)

In [188]:
%time fib_rec_cy(35)

CPU times: user 2.97 s, sys: 19.2 ms, total: 2.99 s
Wall time: 2.96 s


9227465

In [193]:
from functools import lru_cache as cache

In [194]:
@cache(maxsize=None)
def fib_rec_py2(n):
    if n < 2:
        return n
    else:
        return fib_rec_py2(n-1) + fib_rec_py2(n-2)

In [195]:
%time fib_rec_py2(35)

CPU times: user 146 μs, sys: 0 ns, total: 146 μs
Wall time: 205 μs


9227465

In [196]:
%time fib_rec_py2(80)

CPU times: user 111 μs, sys: 1 μs, total: 112 μs
Wall time: 124 μs


23416728348467685

### Iterative Algorithm 

In [197]:
def fib_it_py(n):
    x , y = 0 , 1
    for i in range(1, n+1):
        x , y = y , x + y
    return x

In [199]:
%time fib_it_py(80)

CPU times: user 35 μs, sys: 0 ns, total: 35 μs
Wall time: 45.3 μs


23416728348467685

In [200]:
fib_it_nb = numba.jit(fib_it_py)

In [202]:
%time fib_it_nb(80)

CPU times: user 210 ms, sys: 16 ms, total: 226 ms
Wall time: 224 ms


23416728348467685

In [204]:
%time fib_it_nb(80)

CPU times: user 21 μs, sys: 0 ns, total: 21 μs
Wall time: 30 μs


23416728348467685

In [205]:
%%cython
def fib_it_cy1(int n):
    cdef long i
    cdef long x = 0, y = 1
    for i in range(1, n+1):
        x, y = y, x+y
    return x

In [208]:
%time fib_it_cy1(80)

CPU times: user 15 μs, sys: 0 ns, total: 15 μs
Wall time: 24.8 μs


23416728348467685

In [212]:
%%time

fn = fib_rec_py2(150)
print(fn)

9969216677189303386214405760200
CPU times: user 337 μs, sys: 0 ns, total: 337 μs
Wall time: 300 μs


In [213]:
fn.bit_length()

103

In [215]:
%%time

fn = fib_it_nb(150)
print(fn)

6792540214324356296
CPU times: user 390 μs, sys: 0 ns, total: 390 μs
Wall time: 360 μs


In [216]:
fn.bit_length()

63

In [217]:
%%time

fn = fib_it_cy1(150)
print(fn)

6792540214324356296
CPU times: user 625 μs, sys: 996 μs, total: 1.62 ms
Wall time: 1.27 ms


In [218]:
fn.bit_length()

63

In [219]:
%%cython
cdef extern from *:
    ctypedef int int128 "__int128_t"
def fib_it_cy2(int n):
    cdef int128 i
    cdef int128 x = 0, y = 1
    for i in range(1, n+1):
        x,y = y, x+ y
    return x

In [220]:
%%time
fn = fib_it_cy2(150)
print(fn)

9969216677189303386214405760200
CPU times: user 77 μs, sys: 988 μs, total: 1.07 ms
Wall time: 452 μs


In [221]:
fn.bit_length()

103